In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/input/datasets/brycecf/give-me-some-credit-dataset/cs-training.csv', index_col=0)
print("Original shape:", df.shape)

Original shape: (150000, 11)


In [2]:
def engineer_features(df):
    df = df.copy()

    # Fill missing values
    df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
    df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

    # Cap extreme outliers
    df['RevolvingUtilizationOfUnsecuredLines'] = df[
        'RevolvingUtilizationOfUnsecuredLines'].clip(0, 1)
    df['DebtRatio'] = df['DebtRatio'].clip(0, 10)
    df['MonthlyIncome'] = df['MonthlyIncome'].clip(0, df['MonthlyIncome'].quantile(0.99))

    # New features
    df['MonthlyDebt'] = df['MonthlyIncome'] * df['DebtRatio']
    df['IncomePerDependent'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

    df['TotalLatePayments'] = (
        df['NumberOfTime30-59DaysPastDueNotWorse'] +
        df['NumberOfTime60-89DaysPastDueNotWorse'] +
        df['NumberOfTimes90DaysLate']
    )

    df['AnyLatePayment'] = (df['TotalLatePayments'] > 0).astype(int)
    df['SevereDelinquency'] = (df['NumberOfTimes90DaysLate'] > 0).astype(int)

    df['HighUtilization'] = (
        df['RevolvingUtilizationOfUnsecuredLines'] > 0.75).astype(int)
    df['LowIncome'] = (df['MonthlyIncome'] < 2000).astype(int)
    df['HighDebtRatio'] = (df['DebtRatio'] > 0.4).astype(int)

    df['CreditLinesPerYear'] = df['NumberOfOpenCreditLinesAndLoans'] / (df['age'] + 1)
    df['RealEstateLinesRatio'] = df['NumberRealEstateLoansOrLines'] / (
        df['NumberOfOpenCreditLinesAndLoans'] + 1)

    # Age buckets (ordinal encoded)
    df['AgeBucket'] = pd.cut(
        df['age'],
        bins=[0, 25, 35, 50, 65, 120],
        labels=[0, 1, 2, 3, 4]
    ).astype(float)

    return df

df_eng = engineer_features(df)
print("Engineered shape:", df_eng.shape)
print("\nNew columns added:")
new_cols = [
    'MonthlyDebt', 'IncomePerDependent', 'TotalLatePayments',
    'AnyLatePayment', 'SevereDelinquency', 'HighUtilization',
    'LowIncome', 'HighDebtRatio', 'CreditLinesPerYear',
    'RealEstateLinesRatio', 'AgeBucket'
]
df_eng[new_cols].describe().round(3)

Engineered shape: (150000, 22)

New columns added:


,MonthlyDebt,IncomePerDependent,TotalLatePayments,AnyLatePayment,SevereDelinquency,HighUtilization,LowIncome,HighDebtRatio,CreditLinesPerYear,RealEstateLinesRatio,AgeBucket
count,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,149999.000
mean,11523.963,4422.368,0.927,0.202,0.056,0.183,0.073,0.463,0.166,0.105,2.560
std,20132.244,3180.151,12.466,0.402,0.229,0.387,0.259,0.499,0.104,0.111,0.997
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,762.705,2151.000,0.000,0.000,0.000,0.000,0.000,0.000,0.092,0.000,2.000
50%,2097.856,4000.000,0.000,0.000,0.000,0.000,0.000,0.000,0.148,0.091,3.000
75%,4758.371,5400.000,0.000,0.000,0.000,0.000,0.000,1.000,0.220,0.167,3.000
max,203612.168,23000.000,294.000,1.000,1.000,1.000,1.000,1.000,6.000,0.915,4.000


In [3]:
# Check missing values after engineering
print("Missing after engineering:")
print(df_eng.isnull().sum()[df_eng.isnull().sum() > 0])

# Save for next notebooks
df_eng.to_csv('/kaggle/working/data_engineered.csv', index=False)
print("\nSaved to /kaggle/working/data_engineered.csv")
print("Final shape:", df_eng.shape)

Missing after engineering:
AgeBucket    1
dtype: int64

Saved to /kaggle/working/data_engineered.csv
Final shape: (150000, 22)
